# 02 · Pre-training a tiny GPT from scratch (nanoGPT style)

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~1 minute · **GPU:** optional (full-size config)

Every frontier LLM is, at its core, the same recipe: a decoder-only transformer trained to predict the next token on a
very large pile of text. In this notebook we run that recipe end to end at toy scale: a character-level GPT with
fewer than 1M parameters, trained for about a minute on 300 KB of Shakespeare. We build causal self-attention by hand,
check it against PyTorch's fused kernel, then train, evaluate and sample. The point is to see each moving part
(initialisation, weight tying, AdamW with selective weight decay, warmup and cosine decay, gradient clipping) and
watch the loss go from "uniform guess" to "plausible Elizabethan gibberish".

> **Source & license credit.** The model and training loop in [`research/src/tiny_gpt.py`](../src/tiny_gpt.py) are
> adapted and condensed from **Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) (MIT License)**
> (`model.py`, `train.py`, and the `shakespeare_char` config). The step-by-step self-attention derivation follows the
> structure of Karpathy's *"Let's build GPT"* lecture ([ng-video-lecture](https://github.com/karpathy/ng-video-lecture), MIT);
> the code here is written fresh. Data: a ~300 KB public-domain Shakespeare excerpt (via
> [char-rnn tinyshakespeare](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare)).
> See [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Turn raw text into `(input, target)` training batches for next-token prediction.
2. Derive **causal self-attention** from "average the past" and verify it against `F.scaled_dot_product_attention`.
3. Assemble a GPT: token + position embeddings → pre-LN transformer blocks → tied unembedding.
4. Sanity-check a fresh model (initial loss ≈ ln |V|) and train it with a modern recipe.
5. Read loss curves (train vs. val), convert loss to **bits per character**, and sample with temperature / top-k.

## Background

* Vaswani et al., *Attention Is All You Need*, 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
* Radford et al., *Improving Language Understanding by Generative Pre-Training* (GPT-1), OpenAI 2018. [PDF](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
* Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2), OpenAI 2019. [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
* Brown et al., *Language Models are Few-Shot Learners* (GPT-3), 2020. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
* Xiong et al., *On Layer Normalization in the Transformer Architecture* (why pre-LN), 2020. [arXiv:2002.04745](https://arxiv.org/abs/2002.04745)
* Press & Wolf, *Using the Output Embedding to Improve Language Models* (weight tying), 2017. [arXiv:1608.05859](https://arxiv.org/abs/1608.05859)
* Loshchilov & Hutter, *Decoupled Weight Decay Regularization* (AdamW), 2019. [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)

**Objective.** Given tokens $x_1,\dots,x_T$, minimise the average negative log-likelihood
$\mathcal{L} = -\frac{1}{T}\sum_t \log p_\theta(x_{t+1}\mid x_{\le t})$. A model that knows nothing predicts uniformly,
so the initial loss should be $\ln |V|$ (≈ 4.13 nats for the 62 distinct characters in our excerpt).

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())

In [ ]:
import math, time, inspect
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tiny_gpt import GPT, GPTConfig, CharTokenizer, train_val_split, get_batch, train, estimate_loss

device = CFG.device
text = read_text("tiny_shakespeare.txt")
tok = CharTokenizer(text)
data = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = train_val_split(data, 0.9)
print(f"vocab size {tok.vocab_size}: {''.join(tok.chars)!r}")
print(f"{len(train_data):,} train / {len(val_data):,} val characters")
print("uniform-guess loss = ln(V) =", round(math.log(tok.vocab_size), 3))

## 1 · Batches for next-token prediction

A training example is a random window of `block_size` characters; the target is the same window shifted by one.
One window therefore holds `block_size` prediction problems with context lengths 1 … `block_size`.

In [ ]:
xb, yb = get_batch(train_data, block_size=8, batch_size=2, generator=torch.Generator().manual_seed(0))
for t in range(4):
    print(f"context {tok.decode(xb[0, :t + 1])!r:12} -> target {tok.decode([yb[0, t]])!r}")

## 2 · Self-attention, derived

**Step 1, average the past.** The simplest way to let token *t* see tokens ≤ *t* is to average them. A lower-triangular
matrix of weights does exactly that in one matmul.

**Step 2, make the weights data-dependent.** Each token emits a query $q$, a key $k$ and a value $v$. Affinities are
$qk^\top/\sqrt{d_k}$ (the $\sqrt{d_k}$ keeps the softmax from saturating at init), future positions are masked to
$-\infty$, and a softmax turns each row into a distribution over the past:
$$\mathrm{Attn}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M_{\text{causal}}\right)V.$$

In [ ]:
torch.manual_seed(0)
B, T, C = 1, 5, 8
x = torch.randn(B, T, C)

# step 1: uniform average over the past via a masked softmax of zeros
mask = torch.tril(torch.ones(T, T))
w = torch.zeros(T, T).masked_fill(mask == 0, float("-inf")).softmax(dim=-1)
print("uniform causal weights:\n", w.numpy().round(2))

# step 2: one attention head, by hand
head_size = 4
q_proj, k_proj, v_proj = (nn.Linear(C, head_size, bias=False) for _ in range(3))
q, k, v = q_proj(x), k_proj(x), v_proj(x)
scores = q @ k.transpose(-2, -1) / math.sqrt(head_size)
scores = scores.masked_fill(mask == 0, float("-inf"))
att = scores.softmax(dim=-1)
out_manual = att @ v
out_fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("data-dependent weights (row t attends to positions <= t):\n", att[0].detach().numpy().round(2))
print("matches F.scaled_dot_product_attention:", torch.allclose(out_manual, out_fused, atol=1e-6))

## 3 · The GPT block and model

Multi-head attention runs several heads in parallel and concatenates them. A block alternates **communication**
(attention, tokens exchange information) and **computation** (a per-token MLP), each wrapped in a residual connection
with a pre-LayerNorm. The full model lives in `src/tiny_gpt.py`; here is the block:

In [ ]:
import tiny_gpt
print(inspect.getsource(tiny_gpt.Block))
print(inspect.getsource(tiny_gpt.CausalSelfAttention.forward))

In [ ]:
model_cfg = GPTConfig(
    vocab_size=tok.vocab_size,
    block_size=pick(128, 256),
    n_layer=pick(4, 6),
    n_head=pick(4, 6),
    n_embd=pick(128, 384),
    dropout=pick(0.0, 0.2),
)
model = GPT(model_cfg).to(device)
print(model_cfg)
print(f"parameters: {model.num_params(non_embedding=False):,} total, {model.num_params():,} non-embedding")

### Sanity check before training

At initialisation the logits are near zero, so the loss should be close to $\ln |V| = \ln 62 \approx 4.13$. If it is much
higher, the init is too large and the model starts out confidently wrong (a classic silent bug).

In [ ]:
with torch.no_grad():
    _, loss0 = model(*get_batch(train_data, model_cfg.block_size, 32, device))
print(f"initial loss {loss0.item():.3f} vs ln(V) = {math.log(tok.vocab_size):.3f}")

## 4 · Train

Recipe (from nanoGPT): AdamW with β = (0.9, 0.95), weight decay only on 2-D weight matrices, a linear **warmup** then
**cosine decay**, and gradient-norm clipping at 1.0. CPU default: ~1,200 steps × 32 sequences × 128 characters ≈ 4.9M
characters, about 18 passes over the training split. Full size (the nanoGPT `shakespeare_char` config) is
**GPU-recommended**.

In [ ]:
TRAIN = dict(
    max_steps=pick(1200, 5000),
    batch_size=pick(32, 64),
    lr=pick(2e-3, 1e-3),
    warmup=pick(60, 100),
    eval_every=pick(200, 500),
    eval_iters=pick(10, 50),
)
if FULL_SIZE and not CFG.has_gpu:
    print("Note: FULL_SIZE on CPU will take a long time (tens of minutes or more).")
print(TRAIN)
hist = train(model, train_data, val_data, device=device, seed=CFG.seed, **TRAIN)
print(f"trained in {hist['seconds']:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(len(hist["batch_loss"])), hist["batch_loss"], alpha=0.3, label="batch loss")
ax.plot(hist["step"], hist["train_loss"], "o-", label="train (eval)")
ax.plot(hist["step"], hist["val_loss"], "o-", label="val (eval)")
ax.axhline(math.log(tok.vocab_size), ls="--", c="gray", lw=1, label="uniform guess")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy (nats)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

final_val = hist["val_loss"][-1]
print(f"final val loss {final_val:.3f} nats = {final_val / math.log(2):.3f} bits per character")
print(f"train/val gap: {hist['val_loss'][-1] - hist['train_loss'][-1]:+.3f}")

**Reading the curve.** Bits per character (bpc) is the loss divided by ln 2. Its theoretical floor is the entropy of
the text. Shannon (1951) estimated English at roughly 0.6–1.3 bpc for human predictors. For reference, nanoGPT's
`shakespeare_char` config reports val ≈ 1.47 nats on the full 1 MB dataset with a ~10.7M-parameter model; we use a
quarter of the data and a model ~13× smaller, so expect a noticeably higher number. A widening train/val gap means the model has started to memorise our small corpus; dropout
(used in the full-size config) and more data are the standard remedies.

## 5 · Sample

In [ ]:
model.eval()
prompt = "ROMEO:\n"
for temperature in (0.5, 1.0):
    torch.manual_seed(CFG.seed)
    idx = torch.tensor([tok.encode(prompt)], device=device)
    out = model.generate(idx, max_new_tokens=300, temperature=temperature, top_k=20)
    print(f"--- temperature {temperature} ---")
    print(tok.decode(out[0].tolist()))
    print()

Low temperature gives repetitive but well-formed text; high temperature gives more variety and more misspellings.
Even at this scale the model has learned the *format* (speaker names in capitals followed by a colon, line breaks,
verse-like line lengths) long before it learns meaning. That ordering, from format to syntax to semantics, shows up
in big models too.

## 6 · Save a checkpoint

In [ ]:
ckpt_path = CFG.output_dir / "tiny_gpt_shakespeare.pt"
torch.save({"model": model.state_dict(), "cfg": model_cfg.__dict__, "chars": tok.chars}, ckpt_path)
print(f"saved {ckpt_path.name} ({ckpt_path.stat().st_size / 1e6:.1f} MB) to research/outputs/ (git-ignored)")

## 7 · (Optional, heavy) the full nanoGPT `shakespeare_char` run

The `FULL_SIZE` config above is the nanoGPT `shakespeare_char` setting (6 layers, 6 heads, 384-d, 256 context,
dropout 0.2, ~10.7M parameters). On a free Colab T4 it takes roughly 5–10 minutes for 5,000 steps. On CPU it is not
recommended. The cell below only runs when `FULL_SIZE=True` **and** a GPU is present, and adds `torch.compile` for speed.

In [ ]:
if heavy_cell("Full nanoGPT shakespeare_char run with torch.compile", needs_gpu=True):
    big = GPT(model_cfg).to(device)
    big = torch.compile(big)
    hist_big = train(big, train_data, val_data, device=device, **TRAIN)
    print("final val loss", hist_big["val_loss"][-1])

## Your turn

1. **Ablate positional embeddings.** Zero out `wpe` (or remove it) and retrain. Attention is permutation-invariant,
   so what does the model lose? Look at the samples.
2. **Ablate the residual stream.** Remove the `x +` in `Block.forward`. How does the 4-layer model train now?
3. **Learning-rate sweep.** Try `lr ∈ {3e-4, 1e-3, 3e-3, 1e-2}` for 400 steps each and plot final val loss vs. lr.
   Where is the edge of stability?
4. **Use your BPE tokenizer** from notebook 01 instead of characters, and compare models by **bits per byte**
   (total nats / total bytes / ln 2), not loss per token.

## Beyond the paper — a mini research question

*How much does warmup matter at tiny scale?* Large-scale training almost always uses warmup, motivated by Adam's
early-step variance (Liu et al., *On the Variance of the Adaptive Learning Rate and Beyond*, 2020,
[arXiv:1908.03265](https://arxiv.org/abs/1908.03265)). Run the same model with `warmup ∈ {1, 10, 60, 200}` and
3 seeds each. Does the benefit of warmup grow with learning rate? With depth? Report the mean ± std of final val loss,
and be careful about claiming differences smaller than the seed-to-seed noise.

## References

* Vaswani et al. (2017). *Attention Is All You Need.* [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
* Radford et al. (2018, 2019). GPT-1 / GPT-2 technical reports (OpenAI).
* Brown et al. (2020). *Language Models are Few-Shot Learners.* [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
* Xiong et al. (2020). *On Layer Normalization in the Transformer Architecture.* [arXiv:2002.04745](https://arxiv.org/abs/2002.04745)
* Press & Wolf (2017). *Using the Output Embedding to Improve Language Models.* [arXiv:1608.05859](https://arxiv.org/abs/1608.05859)
* Loshchilov & Hutter (2019). *Decoupled Weight Decay Regularization.* [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
* Shannon, C. (1951). *Prediction and Entropy of Printed English.* Bell System Technical Journal.
* Karpathy, A. *nanoGPT* (MIT) — https://github.com/karpathy/nanoGPT